# Submission Builder

Creates a valid competition ZIP: `solution.py` + `model.pkl` + `requirements.txt`

**Fixes vs original solution.py:**
- Predictions are integers 0-9 (brief requirement)
- Model saved as `model.pkl` (.zip not an allowed extension)
- `preprocess()` runs from raw CSV (no embedded test_clean.csv)
- ZIP has exactly 3 files at root (flat structure)

**Best config from modelling.ipynb:** 1 model, 150 trees, est. score ~0.553

## 1. Imports & Paths

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib, os, zipfile
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
import warnings; warnings.filterwarnings('ignore')

SEED, SMOOTHING = 42, 300
DATA_DIR = '../Data'
SUB_DIR  = '../submission'
os.makedirs(SUB_DIR, exist_ok=True)

TARGET = 'Purchased_Coverage_Bundle'
BUNDLE_MAP = {0:'Auto_Comprehensive',1:'Auto_Liability_Basic',2:'Basic_Health',
              3:'Family_Comprehensive',4:'Health_Dental_Vision',5:'Home_Premium',
              6:'Home_Standard',7:'Premium_Health_Life',8:'Renter_Basic',9:'Renter_Premium'}

print('LightGBM:', lgb.__version__)

LightGBM: 4.6.0


## 2. Load Data

In [2]:
train     = pd.read_csv(os.path.join(DATA_DIR, 'train_clean.csv'))
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

X = train.drop(columns=[TARGET])
y = train[TARGET]
FEATURE_COLS = X.columns.tolist()

print('Train clean:', train.shape)
print('Test  raw:  ', test_raw.shape)
print('Features:   ', len(FEATURE_COLS))
print()
print(y.value_counts().sort_index().to_string())

Train clean: (60868, 45)
Test  raw:   (15218, 28)
Features:    44

Purchased_Coverage_Bundle
0      823
1     1625
2    36136
3     4831
4    13958
5      479
6      719
7     2286
8        6
9        5


## 3. Full-Train Target Encoding Maps

In [3]:
GLOBAL_MEAN = float(train_raw[TARGET].mean())

def build_enc_map(series, target_series):
    df_tmp = pd.DataFrame({'cat': series.fillna(-1), 'tgt': target_series})
    agg = df_tmp.groupby('cat')['tgt'].agg(['mean', 'count'])
    agg['enc'] = (agg['count']*agg['mean'] + SMOOTHING*GLOBAL_MEAN) / (agg['count']+SMOOTHING)
    return agg['enc'].to_dict()

BROKER_ENC_MAP   = build_enc_map(train_raw['Broker_ID'],   train_raw[TARGET])
REGION_ENC_MAP   = build_enc_map(train_raw['Region_Code'], train_raw[TARGET])
EMPLOYER_ENC_MAP = build_enc_map(train_raw['Employer_ID'], train_raw[TARGET])

print(f'Broker  map: {len(BROKER_ENC_MAP)} entries')
print(f'Region  map: {len(REGION_ENC_MAP)} entries')
print(f'Employer map: {len(EMPLOYER_ENC_MAP)} entries')
print(f'Global mean: {GLOBAL_MEAN:.4f}')

Broker  map: 316 entries
Region  map: 167 entries
Employer map: 310 entries
Global mean: 2.7441


## 4. Resample + Train (150 Trees)

In [4]:
def resample(X_tr, y_tr, seed=SEED):
    vc = pd.Series(y_tr).value_counts()
    ros_s = {c: 50 for c in [8, 9] if c in vc.index and vc[c] < 50}
    if ros_s:
        X_tr, y_tr = RandomOverSampler(sampling_strategy=ros_s, random_state=seed).fit_resample(X_tr, y_tr)
    vc = pd.Series(y_tr).value_counts()
    sm_s = {c: 2000 for c in [0, 5, 6] if c in vc.index and vc[c] < 2000}
    if sm_s:
        X_tr, y_tr = SMOTE(sampling_strategy=sm_s, k_neighbors=5, random_state=seed).fit_resample(X_tr, y_tr)
    return X_tr, y_tr

X_bal, y_bal = resample(X.values, y.values)
print('After resampling:', pd.Series(y_bal).value_counts().sort_index().to_string())

model = lgb.LGBMClassifier(
    objective='multiclass', num_class=10, metric='multi_logloss',
    class_weight='balanced', n_estimators=150, learning_rate=0.05,
    max_depth=7, num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=1, verbose=-1,
)
model.fit(X_bal, y_bal)
booster = model.booster_
print(f'\nTrained. Trees per class: {booster.num_trees()//10}')

After resampling: 0     2000
1     1625
2    36136
3     4831
4    13958
5     2000
6     2000
7     2286
8       50
9       50



Trained. Trees per class: 150


## 5. OOF-Tuned Thresholds

In [5]:
# Thresholds from modelling.ipynb OOF analysis — tuned on held-out validation data
THRESHOLDS = np.array([
    0.7073,  # 0 Auto_Comprehensive
    0.6834,  # 1 Auto_Liability_Basic
    0.2681,  # 2 Basic_Health       (suppress over-prediction)
    0.4714,  # 3 Family_Comprehensive
    0.6065,  # 4 Health_Dental_Vision
    0.5488,  # 5 Home_Premium
    0.5162,  # 6 Home_Standard
    0.3874,  # 7 Premium_Health_Life
    0.4056,  # 8 Renter_Basic
    0.6854,  # 9 Renter_Premium
])

train_proba = booster.predict(X.values)
train_preds = (train_proba / THRESHOLDS).argmax(axis=1)
print(f'Train Macro F1 (tuned, on train — inflated): {f1_score(y, train_preds, average="macro", zero_division=0):.4f}')

Train Macro F1 (tuned, on train — inflated): 0.7435


## 6. Save `model.pkl`

In [6]:
MODEL_PATH = os.path.join(SUB_DIR, 'model.pkl')

payload = {
    'model_str':        booster.model_to_string(),
    'thresholds':       THRESHOLDS,
    'feature_cols':     FEATURE_COLS,
    'broker_enc_map':   BROKER_ENC_MAP,
    'region_enc_map':   REGION_ENC_MAP,
    'employer_enc_map': EMPLOYER_ENC_MAP,
    'global_mean':      GLOBAL_MEAN,
}
joblib.dump(payload, MODEL_PATH, compress=3)

size_mb = os.path.getsize(MODEL_PATH) / (1024*1024)
print(f'Saved model.pkl -> {size_mb:.2f} MB')
print(f'Size penalty: max(0.5, 1-{size_mb:.2f}/200) = {max(0.5,1-size_mb/200):.4f}')

Saved model.pkl -> 3.20 MB
Size penalty: max(0.5, 1-3.20/200) = 0.9840


## 7. Write `solution.py`

In [7]:
SOL_PATH = os.path.join(SUB_DIR, 'solution.py')
SOLUTION_PY = open('/tmp/solution_template.py').read()
with open(SOL_PATH, 'w') as f:
    f.write(SOLUTION_PY)
print('Written solution.py —', len(SOLUTION_PY.splitlines()), 'lines')

Written solution.py — 137 lines


## 8. Write `requirements.txt`

In [8]:
REQ_PATH = os.path.join(SUB_DIR, 'requirements.txt')
with open(REQ_PATH, 'w') as f:
    f.write('# All required packages are pre-installed in the judge environment.\n')
    f.write('# lightgbm==4.6.0\n')
    f.write('# numpy==1.26.4\n')
    f.write('# pandas==2.1.4\n')
    f.write('# scikit-learn==1.3.2\n')
    f.write('# joblib==1.3.2\n')
print('Written requirements.txt')

Written requirements.txt


## 9. Build `submission.zip`

In [9]:
ZIP_PATH = '../submission.zip'

files_to_zip = [
    ('solution.py',      SOL_PATH),
    ('model.pkl',        MODEL_PATH),
    ('requirements.txt', REQ_PATH),
]

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for arcname, fpath in files_to_zip:
        zf.write(fpath, arcname)

zip_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
print(f'submission.zip -> {zip_mb:.2f} MB')
print('Contents:')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f'  {info.filename:<22s}  {info.file_size/1024:.1f} KB')
print(f'\nSize OK: {zip_mb <= 50}')

submission.zip -> 3.20 MB
Contents:
  solution.py             6.0 KB
  model.pkl               3271.8 KB
  requirements.txt        0.2 KB

Size OK: True


## 10. Validate — Simulate Judge Pipeline

In [10]:
import importlib.util, sys, time

spec = importlib.util.spec_from_file_location('solution', SOL_PATH)
sol  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)

# 1. Preprocess
df_proc = sol.preprocess(test_raw.copy())
print(f'Preprocessed shape: {df_proc.shape} | User_ID present: {"User_ID" in df_proc.columns}')

# 2. Load model (not timed)
t0 = time.perf_counter()
loaded = sol.load_model()
print(f'Load time: {time.perf_counter()-t0:.3f} s (not counted in score)')

# 3. Predict (timed)
runs = []
for _ in range(5):
    t0 = time.perf_counter()
    out = sol.predict(df_proc.copy(), loaded)
    runs.append(time.perf_counter()-t0)
lat = np.median(runs)

print(f'Predict latency: {lat:.3f} s  |  runs: {[f"{r:.3f}" for r in runs]}')
print(f'Output shape: {out.shape}')
print(f'Columns: {list(out.columns)}')
print(f'Dtype of predictions: {out["Purchased_Coverage_Bundle"].dtype}')
print(f'Unique values: {sorted(out["Purchased_Coverage_Bundle"].unique())}')
print()
print(out.head(8).to_string())

Preprocessed shape: (15218, 45) | User_ID present: True


Load time: 0.059 s (not counted in score)


Predict latency: 0.410 s  |  runs: ['0.424', '0.356', '0.384', '0.410', '0.536']
Output shape: (15218, 2)
Columns: ['User_ID', 'Purchased_Coverage_Bundle']
Dtype of predictions: int64
Unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

      User_ID  Purchased_Coverage_Bundle
0  USR_060868                          2
1  USR_060869                          2
2  USR_060870                          2
3  USR_060871                          2
4  USR_060872                          2
5  USR_060873                          4
6  USR_060874                          2
7  USR_060875                          6


## 11. Final Score Estimate

In [11]:
est_f1  = 0.584
s_pen   = max(0.5, 1 - zip_mb / 200)
l_pen   = max(0.5, 1 - lat / 10)
est     = est_f1 * s_pen * l_pen

print('=' * 55)
print('  ESTIMATED COMPETITION SCORE')
print('=' * 55)
print(f'  Est. Macro F1:   {est_f1:.5f} (OOF fold-0, 150 trees)')
print(f'  ZIP size:        {zip_mb:.2f} MB   (pen: {s_pen:.4f})')
print(f'  Predict latency: {lat:.3f} s    (pen: {l_pen:.4f})')
print(f'  EST. FINAL:      {est:.5f}')
print('=' * 55)
print(f'\nSubmission at: {os.path.abspath(ZIP_PATH)}')

  ESTIMATED COMPETITION SCORE
  Est. Macro F1:   0.58400 (OOF fold-0, 150 trees)
  ZIP size:        3.20 MB   (pen: 0.9840)
  Predict latency: 0.410 s    (pen: 0.9590)
  EST. FINAL:      0.55110

Submission at: /home/tesla/Desktop/DataQuest/DataQuest/submission.zip
